This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [ ]:
import great_expectations as gx
context = gx.get_context()
import logging

In [ ]:
logging.basicConfig(level=logging.INFO, force = True)

In [ ]:
## THIS IS REQUIRED FOR THE TECHNICAL VIEW HACK
# TODO handling of credentials not ideal, required for technical view fix
import os

from google.cloud import bigquery
import pandas as pd

sa_credentials_path=os.environ['SA_CREDENTIALS_PATH']
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_credentials_path
client = bigquery.Client()

In [ ]:
gx_temp_schema = "tech_great_expectations"
date_filter = "hour <= '2099-12-31'"

In [ ]:
def is_sharded_table(client, schema_name: str, table_name: str):
    # TODO CHO20230706 Implement more robust logic to determine whether table is sharded
    return(table_name.endswith("_"))


In [ ]:
def partition_enforcement_enabled(client, schema_name: str, table_name: str):
    partition_enforcement_enabled_sql = f"""
  SELECT
    option_value
  FROM
    {schema_name}.INFORMATION_SCHEMA.TABLE_OPTIONS
  WHERE
    table_name = '{table_name}'
  AND 
    option_name = 'require_partition_filter'
  AND 
    option_value = 'true';
    """
    
    # TODO CHO20230622 handle exceptions
    partition_enforcement_enabled_res = pd.read_gbq(
        partition_enforcement_enabled_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )

    # TODO CHO20230622 handle multiple rows returned
    return(partition_enforcement_enabled_res.shape[0] > 0)

In [ ]:
def get_partition_cols(client, schema_name: str, table_name: str):
    get_partition_cols_sql = f"""
  SELECT
    column_name,
    data_type,
    is_hidden
  FROM
    {schema_name}.INFORMATION_SCHEMA.COLUMNS
  WHERE
    table_name = '{table_name}'
  AND
    is_partitioning_column = 'YES';
    """
    
    # TODO CHO20230622 handle exceptions
    get_partition_cols_res = pd.read_gbq(
        get_partition_cols_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )

    # TODO CHO20230622 handle multiple rows returned
    return(get_partition_cols_res)

In [ ]:
def get_incompatible_columns(client, schema_name: str, table_name: str):
    get_incompatible_cols_sql = f"""
  SELECT
    column_name
  FROM
    {schema_name}.INFORMATION_SCHEMA.COLUMNS
  WHERE
    table_name = '{table_name}'
  AND
    data_type NOT IN (
    'DATE',
    'FLOAT64',
    'STRING',
    'INT64',
    'TIMESTAMP',
    'BOOL',
    'BYTES',
    'TIME',
    'GEOGRAPHY',
    'DATETIME'
  );
    """
    
    # TODO CHO20230622 handle exceptions
    get_incompatible_cols_res = pd.read_gbq(
        get_incompatible_cols_sql, 
        project_id='world-fishing-827', 
        dialect='standard'
    )["column_name"].to_numpy()

    return(get_incompatible_cols_res)

In [ ]:
def create_or_replace_tech_view(
    client, 
    dataset_name: str, 
    table_name: str, 
    gx_temp_schema: str, 
    date_filter_sql: str = None,
    sharded_table_select_shards_sql: str = None,
    exclude_incompatible_columns_sql: str = None,
    select_splitter_col_sql: str = None,
    custom_table_filter: str = None
):

    # TODO CHO20230622 validate parameters, e.g. date filter
    # TODO CHO20230622 check first whether view exists before replacing
    clean_table_name = table_name.replace(".", "_")
    fq_view_name = f"world-fishing-827.{gx_temp_schema}.v_unfiltered_{dataset_name}_{clean_table_name}"
    query = f"""
    CREATE OR REPLACE VIEW `{fq_view_name}` AS (
    SELECT *{exclude_incompatible_columns_sql}{select_splitter_col_sql}
    FROM `world-fishing-827.{dataset_name}.{table_name}{sharded_table_select_shards_sql}` {date_filter_sql}
    {custom_table_filter}
    )
    """
    logging.info(query)
    # TODO CHO20230705 handle exceptions better
    resp=client.query(query)
    logging.info(resp.result())
    return(fq_view_name)    

In [ ]:
connection_string = f"""bigquery://world-fishing-827/tech_great_expectations?\
credentials_path={sa_credentials_path}"""

In [ ]:
import yaml

In [ ]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [ ]:
datasource_config.get("project")

In [ ]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

In [ ]:
exclude_incompatible_columns = get_incompatible_columns(None, "pipe_ais_v3_alpha_published","segs_activity_daily")
if len(exclude_incompatible_columns):
    exclude_incompatible_columns_comma_separated=",".join(exclude_incompatible_columns)
    logging.warning(f"Excluding the following incompatible columns: {exclude_incompatible_columns_comma_separated}")
    exclude_incompatible_columns_sql=f" EXCEPT({exclude_incompatible_columns_comma_separated}) "
else:
    exclude_incompatible_columns_sql=''
print(exclude_incompatible_columns_sql)

In [ ]:
def add_datasource(
        client, 
        datasource_name, 
        dataset_name, 
        table_name, 
        partition_by,
        version_number, 
        gx_temp_schema, 
        gx_datasource,
        default_max_date: str = "2099-12-31",
        asset_exists_behaviour: str = ['update', 'skip'][0],
        custom_table_filter: str = ''
    ):

    asset_name=f"{datasource_name}-{version_number}"
    if asset_name in gx_datasource.get_asset_names():
        if asset_exists_behaviour == 'skip':
            logging.info(f"{asset_name} already exists. Skipping!")
            return
        else:
            logging.info(f"{asset_name} already exists. Updating by removing it before adding again!")
            gx_datasource.delete_asset(asset_name)

    exclude_incompatible_columns = get_incompatible_columns(client, dataset_name, table_name)
    if len(exclude_incompatible_columns):
        exclude_incompatible_columns_comma_separated=",".join(exclude_incompatible_columns)
        logging.warn(f"Excluding the following incompatible columns: {exclude_incompatible_columns_comma_separated}")
        exclude_incompatible_columns_sql=f" EXCEPT({exclude_incompatible_columns_comma_separated}) "
    else:
        exclude_incompatible_columns_sql=''

    date_partition_cols = get_partition_cols(client, dataset_name, table_name).query("data_type.isin(['TIMESTAMP', 'DATE'])")
    partition_filter_enforced = partition_enforcement_enabled(client, dataset_name, table_name)
    table_is_sharded = is_sharded_table(client, dataset_name, table_name)

    date_filter_sql=''
    sharded_table_select_shards_sql=''
    select_splitter_col_sql=''
    has_splitter_col=False

    if table_is_sharded:
        select_splitter_col_sql=f", PARSE_DATE('%Y%m%d', _TABLE_SUFFIX) AS SPLITTER_COLUMN"
        has_splitter_col=True
        logging.info(f"""
            Sharded column _TABLE_SUFFIX is hidden and is therefore added to `SELECT *` statement
        """)
        sharded_table_select_shards_sql='*'

    if partition_by is not None:
        select_splitter_col_sql = f', {partition_by} AS SPLITTER_COLUMN'
        has_splitter_col = True
    else:
        if not date_partition_cols.shape[0]:
            logging.warn(f"""
                No date columns found. Proceeding but proceed cautiously to avoid huge query costs!
            """)
        else:
            if date_partition_cols.query("data_type=='TIMESTAMP'").shape[0]:
                splitter_col=date_partition_cols.query("data_type=='TIMESTAMP'")["column_name"][0]
                select_splitter_col_sql = f', DATE({splitter_col}) AS SPLITTER_COLUMN'
            else:
                splitter_col=date_partition_cols["column_name"][0]
                select_splitter_col_sql = f', {splitter_col} AS SPLITTER_COLUMN'

            date_filter_sql = f"WHERE {splitter_col} < '{default_max_date}'"
            has_splitter_col=True
            logging.info(f"""
                Found date or timestamp type column among partition columns.
                Using date filter: {date_filter}
            """)


    gx_view_name = create_or_replace_tech_view(
        client=client, 
        dataset_name=dataset_name, 
        table_name=table_name, 
        gx_temp_schema=gx_temp_schema, 
        date_filter_sql=date_filter_sql,
        sharded_table_select_shards_sql=sharded_table_select_shards_sql,
        exclude_incompatible_columns_sql='',
        select_splitter_col_sql=select_splitter_col_sql,
        custom_table_filter=custom_table_filter
    )

    batch_metadata={
        'datasource_name': datasource_name, 
        'dataset_name': dataset_name, 
        'table_name': table_name, 
        'version_number': version_number
    }
        
    table_asset = gx_datasource.add_query_asset(
        name=asset_name,
        query=f"SELECT * FROM {gx_view_name}",
        batch_metadata=batch_metadata
    )

    # add splitter and sorter if table is sharded or partition column exists    
    if has_splitter_col:
        table_asset.add_splitter_column_value("SPLITTER_COLUMN")
        table_asset.add_sorters([f"-SPLITTER_COLUMN"])

In [ ]:
previous_assets = gx_datasource.get_asset_names()
previous_assets

In [ ]:
for current_datasource in datasource_config.get("datasources"):
    datasource_name = current_datasource.get('name')
    logging.info(f"Adding datasource '{current_datasource.get('name')}'")
    if 'asset_exists_behaviour' in current_datasource:
        asset_exists_behaviour=current_datasource.get('asset_exists_behaviour')
    else:
        asset_exists_behaviour='skip'
    if 'custom_table_filter' in current_datasource:
        custom_table_filter=current_datasource.get('custom_table_filter')
    else:
        custom_table_filter=''
    for current_version_number in current_datasource.get('versions'):
        config_current_version = current_datasource.get('versions').get(current_version_number)
        dataset_name = config_current_version.get('dataset')
        table_name = config_current_version.get('table')
        partition_by = config_current_version.get('partition_by', None)
        if dataset_name in [
            'pipe_production_v20201001', 
            'pipe_ais_v3_alpha_internal', 
            'pipe_ais_v3_alpha_published', 
            'tech_great_expectations',
            'pipe_brazil_production_v20211126'
        ]:
            logging.info(f"Adding version '{current_version_number}'")
            add_datasource(
                client,
                datasource_name,
                dataset_name, 
                table_name, 
                partition_by,
                current_version_number, 
                gx_temp_schema, 
                gx_datasource, 
                asset_exists_behaviour=asset_exists_behaviour,
                custom_table_filter=custom_table_filter
            )


In [ ]:
# print new assets
set(gx_datasource.get_asset_names()).difference(previous_assets)